# Fabric 03 · The cost-optimal hybrid router

The cheapest token is the one you never send. A deterministic pre-filter resolves the **easy majority** (clearly outside 30 days, bedside/IR, documented staged) for free, and escalates only the **ambiguous** cases to the fine-tuned model — the end-state production design.

In [ ]:
# === Dual-mode setup: Microsoft Fabric (Spark + Lakehouse) OR local (pandas + repo files) ===
import os, json
from pathlib import Path

try:
    import notebookutils            # exists ONLY inside Microsoft Fabric
    IN_FABRIC = True
except Exception:
    IN_FABRIC = False

def _find_data_dir():
    here = Path.cwd()
    for c in [here, *here.parents]:
        d = c / 'fine-tuning' / 'data'
        if d.exists():
            return d
    return Path('fine-tuning/data')
DATA_DIR = None if IN_FABRIC else _find_data_dir()

AZURE_OPENAI_ENDPOINT = os.environ.get('AZURE_OPENAI_ENDPOINT', 'https://<your-foundry>.cognitiveservices.azure.com/')
API_VERSION           = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_DEPLOYMENT       = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
TUNED_DEPLOYMENT      = os.environ.get('TUNED_DEPLOYMENT', 'acme-rtor-deployment')

# Entra token for Azure OpenAI: Fabric token broker in-cloud, DefaultAzureCredential locally.
if IN_FABRIC:
    def _token():
        return notebookutils.credentials.getToken('https://cognitiveservices.azure.com')
else:
    from azure.identity import DefaultAzureCredential
    _cred = DefaultAzureCredential()
    def _token():
        return _cred.get_token('https://cognitiveservices.azure.com/.default').token

from openai import AzureOpenAI
client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = _token,
    api_version             = API_VERSION,
)
print('mode    :', 'FABRIC' if IN_FABRIC else 'LOCAL')
print('endpoint:', AZURE_OPENAI_ENDPOINT)
print('models  : base=' + BASE_DEPLOYMENT + '  tuned=' + TUNED_DEPLOYMENT)


In [ ]:
# === The RTOR abstraction prompt + defensive parser (identical to the Foundry labs) ===
import json

RULES_BLOCK = '''
### SPECIFIC ABSTRACTION RULES

Rule 1 - Conflict-resolution order (apply in this EXACT priority; the FIRST match decides):
  1. Planned / staged overrides everything (planned/staged/anticipated/scheduled at index) -> false, even within 30 days.
  2. Unplanned + related complication (bleeding, hematoma, SSI, dehiscence, anastomotic leak, abscess, graft/flap failure) within 30 days -> true.
  3. Unrelated anatomy or new diagnosis -> false, regardless of timing.
  4. Outside the 30-day window -> false.

Rule 2 - Operating-room requirement. Bedside / ICU / IR / endoscopy-suite / clinic procedures do NOT count -> false.

Rule 3 - Evidence requirement. Quote the single most decisive sentence verbatim, then state which rule it triggers.
'''

SYSTEM_PROMPT = (
    'You are a surgical-quality abstraction assistant for Acme Health. Determine whether the '
    'current operative episode is an unplanned Return to the Operating Room (RTOR) for the index '
    'surgery, applying the rules below.\n'
    + RULES_BLOCK +
    '\n### TASK EXECUTION\n'
    '- Read the provided text thoroughly.\n'
    '- Resolve conflicting data using the exact order in Rule 1.\n'
    '- Output ONLY a valid JSON object with exactly two keys: "is_return_to_or" (boolean) and '
    '"evidence" (string citing the exact text used and how it applies to the rules).\n'
    '- No conversational filler. No markdown json fence.'
)

TEMPLATE = '''Patient Timeline:
{patient_timeline_json}

Progress Note Details:
{progress_note_json}

Index Surgery Procedure Description:
{index_surgery_procedure_desc}

Index Surgery Operative Note:
{index_surgery_op_note}

Current Surgery Procedure Description:
{current_surgery_procedure_desc}

Current Surgery Operative Note:
{current_surgery_op_note}

Task: Determine if the current operating note/surgery represents a return to the operating room based strictly on the abstraction rules provided above. Output ONLY the raw JSON object.'''

def build_user_prompt(case):
    return TEMPLATE.format(
        patient_timeline_json          = json.dumps(case.get('patient_timeline', []), indent=2),
        progress_note_json             = json.dumps(case.get('progress_note', {}), indent=2),
        index_surgery_procedure_desc   = case.get('index_surgery_procedure_desc', ''),
        index_surgery_op_note          = case.get('index_surgery_op_note', ''),
        current_surgery_procedure_desc = case.get('current_surgery_procedure_desc', ''),
        current_surgery_op_note        = case.get('current_surgery_op_note', ''),
    )

def safe_parse(val):
    '''Extract JSON from the model response, stripping stray markdown fences.'''
    try:
        clean = str(val).strip()
        if clean.startswith('```'):
            clean = clean.strip('`')
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean.strip())
    except Exception as e:
        return {'is_return_to_or': None, 'evidence': f'Parse Error: {e} | Raw: {val}'}

print('prompt + parser ready')


In [ ]:
# === Load the eval cases (full dicts incl. gold_*). Fabric -> Lakehouse table; local -> jsonl ===
def load_eval():
    if IN_FABRIC:
        rows = spark.read.table('surgical_episodes').toPandas().to_dict('records')
        return [json.loads(r['case_json']) for r in rows]
    p = DATA_DIR / 'rtor_eval.jsonl'
    assert p.exists(), f'missing {p} -- run the Foundry clinical Lab 00 first'
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]

EVAL = load_eval()
print('eval cases:', len(EVAL))


In [ ]:
# === Persist / load predictions so the eval notebook can compare every approach ===
def save_preds(name, preds):
    if IN_FABRIC:
        import pandas as pd
        (spark.createDataFrame(pd.DataFrame(preds))
            .write.format('delta').mode('overwrite').saveAsTable(f'preds_{name}'))
    else:
        (DATA_DIR / f'preds_{name}.json').write_text(json.dumps(preds), encoding='utf-8')
    print(f'saved preds -> {name} ({len(preds)})')

def load_preds(name):
    if IN_FABRIC:
        return spark.read.table(f'preds_{name}').toPandas().to_dict('records')
    return json.loads((DATA_DIR / f'preds_{name}.json').read_text(encoding='utf-8'))


In [ ]:
# === Shared scoring: classification metrics + LLM-as-judge evidence groundedness ===
def score(preds):
    tp = sum(1 for p in preds if p['gold'] and p['pred'] is True)
    tn = sum(1 for p in preds if (not p['gold']) and p['pred'] is False)
    fp = sum(1 for p in preds if (not p['gold']) and p['pred'] is True)
    fn = sum(1 for p in preds if p['gold'] and p['pred'] is False)
    un = sum(1 for p in preds if p['pred'] is None)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    acc  = (tp + tn) / len(preds) if preds else 0.0
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn, 'unparsed': un}

def judge_groundedness(preds):
    scored = [p for p in preds if p.get('pred') is not None and p.get('pred_evidence')]
    if not scored:
        return 0.0
    total = 0
    for p in scored:
        sysmsg = ('You grade whether an abstraction citation is well-grounded. Given GOLD evidence and '
                  'MODEL evidence for a surgical Return-to-OR decision, reply JSON {"score": 0 or 1}. '
                  'score=1 means the model quoted a relevant source sentence and named a plausible rule.')
        r = client.chat.completions.create(model=BASE_DEPLOYMENT, temperature=0.0, max_tokens=80,
            response_format={'type': 'json_object'},
            messages=[{'role': 'system', 'content': sysmsg},
                      {'role': 'user', 'content': json.dumps({'gold': p.get('gold_evidence'), 'model': p.get('pred_evidence')})}])
        total += int(safe_parse(r.choices[0].message.content).get('score', 0) or 0)
    return total / len(scored)

def print_board(name, m):
    print(f'[{name}]  acc={m["accuracy"]:.0%}  prec={m["precision"]:.0%}  rec={m["recall"]:.0%}  '
          f'f1={m["f1"]:.0%}  (TP={m["tp"]} TN={m["tn"]} FP={m["fp"]} FN={m["fn"]} unparsed={m["unparsed"]})')


---
## Step 1 — The cheap deterministic pre-filter

Returns a confident decision for the unambiguous cases, or `None` to escalate. It only decides when a rule is *clearly* triggered — it never guesses the hard Rule 1.2 calls.

In [ ]:
import re
PLANNED = re.compile(r'planned|staged|scheduled|second[- ]look|anticipat|delayed (closure|fascial)', re.I)
NON_OR  = re.compile(r'bedside|at the bedside|\bICU\b|interventional radiology|\bIR\b|endoscopy suite|in clinic|clinic-based', re.I)

def current_day_offset(case):
    days = [e.get('day_offset') for e in case.get('patient_timeline', []) if isinstance(e, dict)]
    days = [d for d in days if isinstance(d, (int, float))]
    return max(days) if days else None

def cheap_router(case):
    cur = (case.get('current_surgery_op_note', '') + ' ' + json.dumps(case.get('progress_note', {})))
    idx = case.get('index_surgery_op_note', '')
    d = current_day_offset(case)
    if d is not None and d > 30:
        return False, f'current procedure on day {d} (>30) -> Rule 1.4', 'cheap'
    if NON_OR.search(cur):
        return False, 'documented bedside/ICU/IR/endoscopy -> Rule 2', 'cheap'
    if PLANNED.search(cur) or PLANNED.search(idx):
        return False, 'planned/staged language documented -> Rule 1.1', 'cheap'
    return None, 'ambiguous -> escalate to model', 'model'

print('cheap router ready')


---
## Step 2 — Route: cheap where confident, model otherwise

In [ ]:
ESC_MODEL = TUNED_DEPLOYMENT
try:
    client.chat.completions.create(model=ESC_MODEL, max_tokens=3,
        messages=[{'role': 'user', 'content': 'ping'}])
except Exception:
    ESC_MODEL = BASE_DEPLOYMENT
print('escalation model:', ESC_MODEL)

def llm_classify(case, model):
    r = client.chat.completions.create(model=model, temperature=0.0, max_tokens=300,
        response_format={'type': 'json_object'},
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user',   'content': build_user_prompt(case)}])
    return safe_parse(r.choices[0].message.content)

hybrid_preds = []
cheap_n = esc_n = 0
for c in EVAL:
    dec, reason, path = cheap_router(c)
    if dec is not None:
        cheap_n += 1
        pred, ev = dec, reason
    else:
        esc_n += 1
        p = llm_classify(c, ESC_MODEL)
        pred, ev = p.get('is_return_to_or'), p.get('evidence')
    hybrid_preds.append({'case_id': c['case_id'], 'gold': bool(c['gold_is_return_to_or']),
                         'pred': pred, 'pred_evidence': ev, 'gold_evidence': c.get('gold_evidence'),
                         'path': path})
save_preds('hybrid', hybrid_preds)
print(f'cheap path: {cheap_n}/{len(EVAL)} ({cheap_n/len(EVAL):.0%})   escalated to model: {esc_n}')


---
## Step 3 — The cost lever

In [ ]:
AVG_TOKENS_PER_CALL = 700   # rules + case, order of magnitude
print(f'LLM calls avoided: {cheap_n} of {len(EVAL)} ({cheap_n/len(EVAL):.0%})')
print(f'~{cheap_n * AVG_TOKENS_PER_CALL:,} tokens/batch avoided at this sample size.')
print('At production volume (charts/night), the cheap path is the dominant cost lever.')


---
## Step 4 — Did routing keep quality?

In [ ]:
m = score(hybrid_preds)
print_board('hybrid router', m)
print('evidence groundedness:', f"{judge_groundedness(hybrid_preds):.0%}")


---
## Takeaways

- The router trades a little build complexity for a large **cost reduction** — the easy cases never touch the model.
- Escalations go to the fine-tuned model, so the *hard* cases still get the best answer.
- Next: **Fabric 04** scores all four approaches on one board and logs them to MLflow.